In [1]:
import numpy as np
from math import cos, sin, sqrt

In [2]:
from ahrs.filters import Mahony
from ahrs.filters import Madgwick

from ahrs.common import Quaternion

# RTDTransformer

In [ ]:
#LOCAL_ACC_GRAVITY = 9.80199 

class RTDTransformer:
    def __init__(self, aTickZero: int = 0): #e.g. aDataIn = {'tick': 3, metrics: {'metric_1': 2.1, 'metric_2': 2.2}}
        self.__ct_apply = 0
        self.__tick_pre = aTickZero

    
    def reset(self) -> None:
        self.__ct_apply = 0
    

    @property
    def transformed_count(self) -> int:
        return self.__ct_apply
    
    @property
    def tick_pre(self) -> int:
        return self.__tick_pre

    
    def _calc_dt(self, aNewTick: int) -> float:
        return (aNewTick-self.__tick_pre)*1e-6 # convert microseconds to seconds   
        
    def _iterate_angle(self, aPre, aOmiga, aDt):
        return aPre + aOmiga*aDt

        
    def _transform(self, aDataIn: dict) -> dict: #baseclass just pass the input along
        return aDataIn    

                
    def apply(self, aDataIn: dict) -> dict: #e.g. aDataIn = {'tick': 3, metrics: {'metric_1': 2.1, 'metric_2': 2.2}}
        dataOut = self._transform(aDataIn)
       
        self.__tick_pre = aDataIn['tick']
        self.__ct_apply +=1
        return dataOut

# Calibartors

In [7]:
class CalibrateWithSphereAlg(RTDTransformer):
    # This is calibration based for my sensor
    SCALE_X = 9.587101943913872
    SCALE_Y = 10.032382214088624
    SCALE_Z = 10.01370771311872    
    OFFSET_X = 0.14961055101095233
    OFFSET_Y = -0.13218344034859986
    OFFSET_Z = -1.7504601956794568    
    GYRO_BIAS_X = -0.05019117260446154
    GYRO_BIAS_Y = 0.009999999776482582
    GYRO_BIAS_Z = -0.029999999329447746
    

    def __init__(self): 
        super().__init__()


    def _transform(self, aDataIn: dict) -> dict:
        return {'tick': aDataIn['tick'], 
                'acc_x': (aDataIn['AccX'] - self.SCALE_X)/self.SCALE_X, 
                'acc_y': (aDataIn['AccY'] - self.SCALE_Y)/self.SCALE_Y,                 
                'acc_z': (aDataIn['AccZ'] - self.SCALE_Z)/self.SCALE_Z,   
                'gyr_x': aDataIn['GyrX'] - self.GYRO_BIAS_X,                                                                      
                'gyr_y': aDataIn['GyrY'] - self.GYRO_BIAS_Y,       
                'gyr_z': aDataIn['GyrZ'] - self.GYRO_BIAS_Z                       
               }        

# Attitude calculator

## Attitude base

In [5]:
class RTDTransformerAttitude(RTDTransformer):
    Q0 = Quaternion([1., 0., 0., 0.])
    
    def __init__(self, TickZero: int = 0): 
        super().__init__(TickZero)


    def extract_imu(self, aDataIn: dict):
        return {'dt':self._calc_dt(aDataIn['tick']), 
                'acc':np.array([aDataIn['acc_x'], aDataIn['acc_y'], aDataIn['acc_z']]),
                'gyr':np.array([aDataIn['gyr_x'], aDataIn['gyr_y'], aDataIn['gyr_z']])
               }


    def pack_axang_by_q(self, aTick, aQ):
        ret = aQ.to_axang()
        return {'tick': aTick,
                'angle': ret[1],
                'axis_x':ret[0][0], 
                'axis_y':ret[0][1], 
                'axis_z':ret[0][2]} 
        
    def pack_axang_by_euler(self, aTick, aPhi, aTheta, aPsi):     
        q = Quaternion()
        q.from_rpy(np.array([aPhi, aTheta, aPsi])) 
        return self.pack_axang_by_q(aTick, q)

    
    def pack_orientation_by_euler(self, aTick, aPhi, aTheta, aPsi):
        return {'tick': aTick, 
                'phi':aPhi, 'theta':aTheta, 'psi':aPsi}         

    def pack_orientation_by_q(self, aTick, aQ):
        phi, theta, psi = aQ.to_angles()
        return self.pack_orientation_by_euler(aTick, phi, theta, psi)

## Mahony

In [ ]:
class RotationTransformerMahony(RTDTransformerAttitude): 
    def __init__(self): 
        super().__init__()
        
        self._q_prev=None
        self._fiter=Mahony()
    
    def _transform(self, aDataIn: dict) -> dict:
        imu_date = self.extract_imu(aDataIn)

        q_new = RTDTransformerAttitude.Q0 if self._q_prev is None else\
                self._fiter.updateIMU(self._q_prev, 
                                      gyr=imu_date['gyr'], acc=imu_date['acc'], 
                                      dt=imu_date['dt'])

        self._q_prev = q_new
        return self.pack_orientation_by_q(aDataIn['tick'], q_new)

## Madgwick

In [ ]:
class RotationTransformerMadgwick(RTDTransformerAttitude): 
    def __init__(self): 
        super().__init__()
        
        self._q_prev=None
        self._fiter=Madgwick()

    
    def _transform(self, aDataIn: dict) -> dict:
        imu_date = self.extract_imu(aDataIn)

        q_new = RTDTransformerAttitude.Q0 if self._q_prev is None else\
                self._fiter.updateIMU(self._q_prev, 
                                      gyr=imu_date['gyr'], acc=imu_date['acc'], 
                                      dt=imu_date['dt'])

        self._q_prev = q_new
        return self.pack_orientation_by_q(aDataIn['tick'], q_new)    